In [1]:
import numpy as np
import hls4ml
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt
import os

MODEL_PATH = '/Users/giannalongo/Downloads/pioneer/476_Final/cnn_model_final.h5'
OUTPUT_DIR = '/Users/giannalongo/Downloads/pioneer/476_Final'
HLS_OUTPUT = '/Users/giannalongo/Downloads/pioneer/476_Final/hls4ml_project'

# Load the trained model
model = keras.models.load_model(MODEL_PATH)
model.summary()

Model: "sequential_5"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv1d_15 (Conv1D)          (None, 200, 8)            48        
                                                                 
 max_pooling1d_10 (MaxPoolin  (None, 100, 8)           0         
 g1D)                                                            
                                                                 
 conv1d_16 (Conv1D)          (None, 100, 16)           656       
                                                                 
 max_pooling1d_11 (MaxPoolin  (None, 50, 16)           0         
 g1D)                                                            
                                                                 
 conv1d_17 (Conv1D)          (None, 50, 16)            784       
                                                                 
 global_average_pooling1d_5   (None, 16)              

/Users/giannalongo/anaconda3/envs/hls4ml/lib/python3.10/site-packages/hls4ml/converters/__init__.py:27: UserWarning: WARNING: Pytorch converter is not enabled!
  warnings.warn("WARNING: Pytorch converter is not enabled!", stacklevel=1)


In [3]:
config = hls4ml.utils.config_from_keras_model(
    model,
    granularity='name'
)

import pprint
pprint.pprint(config)

Interpreting Sequential
Topology:
Layer name: conv1d_15_input, layer type: InputLayer, input shapes: [[None, 200, 1]], output shape: [None, 200, 1]
Layer name: conv1d_15, layer type: Conv1D, input shapes: [[None, 200, 1]], output shape: [None, 200, 8]
Layer name: max_pooling1d_10, layer type: MaxPooling1D, input shapes: [[None, 200, 8]], output shape: [None, 100, 8]
Layer name: conv1d_16, layer type: Conv1D, input shapes: [[None, 100, 8]], output shape: [None, 100, 16]
Layer name: max_pooling1d_11, layer type: MaxPooling1D, input shapes: [[None, 100, 16]], output shape: [None, 50, 16]
Layer name: conv1d_17, layer type: Conv1D, input shapes: [[None, 50, 16]], output shape: [None, 50, 16]
Layer name: global_average_pooling1d_5, layer type: GlobalAveragePooling1D, input shapes: [[None, 50, 16]], output shape: [None, 16]
Layer name: dense_10, layer type: Dense, input shapes: [[None, 16]], output shape: [None, 32]
Layer name: dense_11, layer type: Dense, input shapes: [[None, 32]], output s

In [5]:

# Convert model to HLS
hls_model = hls4ml.converters.convert_from_keras_model(
    model,
    hls_config=config,
    output_dir=HLS_OUTPUT,
    backend='Vitis',
    part='xc7a200tsbg484-1',
    clock_period=10
)

# Compile and check predictions match original model
hls_model.compile()

# Load test data
X_test = np.load(os.path.join(OUTPUT_DIR, 'X_test.npy'))
Y_test = np.load(os.path.join(OUTPUT_DIR, 'Y_test.npy'))

# Compare predictions
y_keras = np.argmax(model.predict(X_test), axis=1)
y_hls   = np.argmax(hls_model.predict(X_test), axis=1)

match = np.mean(y_keras == y_hls)
print(f'Keras vs HLS prediction agreement: {match*100:.2f}%')

Interpreting Sequential
Topology:
Layer name: conv1d_15_input, layer type: InputLayer, input shapes: [[None, 200, 1]], output shape: [None, 200, 1]
Layer name: conv1d_15, layer type: Conv1D, input shapes: [[None, 200, 1]], output shape: [None, 200, 8]
Layer name: max_pooling1d_10, layer type: MaxPooling1D, input shapes: [[None, 200, 8]], output shape: [None, 100, 8]
Layer name: conv1d_16, layer type: Conv1D, input shapes: [[None, 100, 8]], output shape: [None, 100, 16]
Layer name: max_pooling1d_11, layer type: MaxPooling1D, input shapes: [[None, 100, 16]], output shape: [None, 50, 16]
Layer name: conv1d_17, layer type: Conv1D, input shapes: [[None, 50, 16]], output shape: [None, 50, 16]
Layer name: global_average_pooling1d_5, layer type: GlobalAveragePooling1D, input shapes: [[None, 50, 16]], output shape: [None, 16]
Layer name: dense_10, layer type: Dense, input shapes: [[None, 16]], output shape: [None, 32]
Layer name: dense_11, layer type: Dense, input shapes: [[None, 32]], output s

2026-04-30 14:02:40.973013: W tensorflow/tsl/platform/profile_utils/cpu_utils.cc:128] Failed to get CPU frequency: 0 Hz


Keras vs HLS prediction agreement: 80.53%


In [6]:
# Adjust precision for better accuracy
config['Model']['Precision'] = 'fixed<16,6>'

# Give more integer bits to layers that accumulate large values
config['LayerName']['conv1d_15']['Precision']['result'] = 'fixed<20,8>'
config['LayerName']['conv1d_16']['Precision']['result'] = 'fixed<20,8>'
config['LayerName']['conv1d_17']['Precision']['result'] = 'fixed<20,8>'
config['LayerName']['dense_10']['Precision']['result'] = 'fixed<20,8>'
config['LayerName']['dense_11']['Precision']['result'] = 'fixed<20,8>'

import shutil
if os.path.exists(HLS_OUTPUT):
    shutil.rmtree(HLS_OUTPUT)

hls_model = hls4ml.converters.convert_from_keras_model(
    model,
    hls_config=config,
    output_dir=HLS_OUTPUT,
    backend='Vivado',
    part='xc7a200tsbg484-1',
    clock_period=10
)

hls_model.compile()

y_keras = np.argmax(model.predict(X_test), axis=1)
y_hls   = np.argmax(hls_model.predict(X_test), axis=1)

match = np.mean(y_keras == y_hls)
print(f'Keras vs HLS prediction agreement: {match*100:.2f}%')

Interpreting Sequential
Topology:
Layer name: conv1d_15_input, layer type: InputLayer, input shapes: [[None, 200, 1]], output shape: [None, 200, 1]
Layer name: conv1d_15, layer type: Conv1D, input shapes: [[None, 200, 1]], output shape: [None, 200, 8]
Layer name: max_pooling1d_10, layer type: MaxPooling1D, input shapes: [[None, 200, 8]], output shape: [None, 100, 8]
Layer name: conv1d_16, layer type: Conv1D, input shapes: [[None, 100, 8]], output shape: [None, 100, 16]
Layer name: max_pooling1d_11, layer type: MaxPooling1D, input shapes: [[None, 100, 16]], output shape: [None, 50, 16]
Layer name: conv1d_17, layer type: Conv1D, input shapes: [[None, 50, 16]], output shape: [None, 50, 16]
Layer name: global_average_pooling1d_5, layer type: GlobalAveragePooling1D, input shapes: [[None, 50, 16]], output shape: [None, 16]
Layer name: dense_10, layer type: Dense, input shapes: [[None, 16]], output shape: [None, 32]
Layer name: dense_11, layer type: Dense, input shapes: [[None, 32]], output s

In [7]:
y_keras = np.argmax(model.predict(X_test), axis=1)
y_hls   = np.argmax(hls_model.predict(X_test), axis=1)

match = np.mean(y_keras == y_hls)
print(f'Keras vs HLS prediction agreement: {match*100:.2f}%')

24/24 [==============================] - 0s 2ms/step
Keras vs HLS prediction agreement: 80.53%


In [8]:
import shutil

# Reset config
config = hls4ml.utils.config_from_keras_model(model, granularity='name')

# Increase total bits and fractional bits across the board
config['Model']['Precision'] = 'fixed<24,8>'

for layer in config['LayerName']:
    for key in config['LayerName'][layer]['Precision']:
        config['LayerName'][layer]['Precision'][key] = 'fixed<24,8>'

# Rebuild
if os.path.exists(HLS_OUTPUT):
    shutil.rmtree(HLS_OUTPUT)

hls_model = hls4ml.converters.convert_from_keras_model(
    model,
    hls_config=config,
    output_dir=HLS_OUTPUT,
    backend='Vivado',
    part='xc7a200tsbg484-1',
    clock_period=10
)

hls_model.compile()

y_keras = np.argmax(model.predict(X_test), axis=1)
y_hls   = np.argmax(hls_model.predict(X_test), axis=1)

match = np.mean(y_keras == y_hls)
print(f'Keras vs HLS prediction agreement: {match*100:.2f}%')

Interpreting Sequential
Topology:
Layer name: conv1d_15_input, layer type: InputLayer, input shapes: [[None, 200, 1]], output shape: [None, 200, 1]
Layer name: conv1d_15, layer type: Conv1D, input shapes: [[None, 200, 1]], output shape: [None, 200, 8]
Layer name: max_pooling1d_10, layer type: MaxPooling1D, input shapes: [[None, 200, 8]], output shape: [None, 100, 8]
Layer name: conv1d_16, layer type: Conv1D, input shapes: [[None, 100, 8]], output shape: [None, 100, 16]
Layer name: max_pooling1d_11, layer type: MaxPooling1D, input shapes: [[None, 100, 16]], output shape: [None, 50, 16]
Layer name: conv1d_17, layer type: Conv1D, input shapes: [[None, 50, 16]], output shape: [None, 50, 16]
Layer name: global_average_pooling1d_5, layer type: GlobalAveragePooling1D, input shapes: [[None, 50, 16]], output shape: [None, 16]
Layer name: dense_10, layer type: Dense, input shapes: [[None, 16]], output shape: [None, 32]
Layer name: dense_11, layer type: Dense, input shapes: [[None, 32]], output s

In [9]:
from sklearn.metrics import classification_report

y_true = np.argmax(Y_test, axis=1)

print('HLS Model Performance:')
print(classification_report(y_true, y_hls, target_names=['Noise', 'Single Peak', 'Double Peak']))

HLS Model Performance:
              precision    recall  f1-score   support

       Noise       0.95      1.00      0.97       252
 Single Peak       0.83      0.90      0.86       253
 Double Peak       0.94      0.80      0.87       245

    accuracy                           0.90       750
   macro avg       0.90      0.90      0.90       750
weighted avg       0.90      0.90      0.90       750



In [10]:
print(f'HLS project location: {HLS_OUTPUT}')

HLS project location: /Users/giannalongo/Downloads/pioneer/476_Final/hls4ml_project
